# Notebook 23 — Knowledge and Reasoning Distillation

    ## Learning objectives

    - Compare sequence, logit, and on-policy distillation
- Build teacher-data and KL objectives
- Measure transferred capability, error, and serving benefit

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = []

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if True and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 23.1 What is transferred

Sequence distillation trains a student on teacher-generated targets; logit distillation matches softened teacher distributions; intermediate-feature methods align representations; on-policy distillation scores student-generated trajectories. Distillation can compress a model, transfer instruction behavior, or teach reasoning and tools. It cannot exceed evidence available from tasks, teachers, environments, and verifiers without additional exploration. Teacher outputs contain correlated factual, stylistic, safety, and calibration errors.


In [ ]:
import torch
t=torch.tensor([[3.,1.,0.]]); s=torch.tensor([[1.,1.,1.]],requires_grad=True)
T=2.; loss=torch.nn.functional.kl_div(torch.log_softmax(s/T,-1),torch.softmax(t/T,-1),reduction="batchmean")*T*T; loss.backward(); print(loss.item(),s.grad)


## 23.2 Temperature and objectives

KL divergence between teacher and student distributions exposes dark knowledge in non-argmax probabilities. Temperature softens distributions; conventional objectives rescale by temperature squared so gradient magnitude remains comparable. Tokenizer mismatch complicates token-level alignment, making sequence targets or character/span alignment safer. Combine distillation with ground-truth loss deliberately and mask prompt/padding positions. Direction of KL changes mode-covering behavior.


In [ ]:
records=[{"prompt":"2+2","teacher":"FINAL: 4","verified":True},{"prompt":"3*7","teacher":"FINAL: 20","verified":False}]
print([r for r in records if r["verified"]])


## 23.3 Reasoning and tool traces

Reasoning distillation uses worked solutions, rejection-sampled correct traces, or teacher trajectories. Verify final outcomes independently and filter invalid intermediate steps where possible. Long polished rationales are not necessarily faithful. For tools, retain schemas, calls, results, recovery examples, and permission denials; never train authorization from model text. On-policy sampling reduces train-test mismatch but costs inference and requires stable teachers or rewards.


In [ ]:
teacher_tokens=10_000_000; student_training=2_000_000; saving_per_request=800; requests=20_000
print("net token-equivalent",requests*saving_per_request-teacher_tokens-student_training)


## 23.4 Evaluation and economics

Compare student with untrained student, teacher, and simpler SFT baselines under fixed prompts. Evaluate target capability, general retention, calibration, safety, pass@k, tool behavior, and teacher-error reproduction. Measure total generation/training cost against lifetime serving savings, artifact size, memory, latency, and throughput. A smaller student may need more tokens to solve reasoning tasks. Preserve teacher revisions, generation settings, filters, data lineage, objective weights, and licenses.


In [ ]:
metrics={"teacher_accuracy":.91,"student_before":.62,"student_after":.84,"latency_ratio":.35}; print(metrics)


## Reference workflow and evidence standard

Treat the notebook as an experiment, not a recipe. State the question, freeze inputs and
success criteria, establish the simplest baseline, change one material factor, and retain raw
outputs needed to diagnose failures. Record model, tokenizer, template, data and code revisions;
hardware and dtype; random seeds; generation or optimization configuration; token counts;
latency and memory; and results by meaningful slice. A demonstration that runs is evidence of
plumbing, not evidence of general capability.

Test boundaries as well as the happy path: empty and maximum-length inputs, malformed records,
multilingual or code text, unavailable dependencies, cancellation, and adversarial content.
Keep credentials in environment or Colab Secrets and never serialize them with artifacts. Pin
remote revisions, review licenses and custom code, validate saved artifacts in a fresh process,
and prefer deterministic validators wherever outputs can be checked mechanically.

Before applying the technique, compare it with prompting, retrieval, a smaller model, or no
model. Report quality together with compute, storage, latency, and operational complexity. Use
held-out data and paired comparisons, disclose uncertainty and negative results, and define a
rollback path. These practices connect low-level understanding to reliable application work.

A useful completion checklist asks four separate questions. Is the mathematical contract clear
enough to predict shapes, masks, reductions, and failure cases? Does the implementation reproduce
a tiny hand-worked or deterministic reference? Does the measured result survive a held-out set,
relevant slices, and an ablation against a simpler baseline? Can another person reload the exact
artifacts and reconstruct the claim from the manifest? Passing only the first two establishes a
tutorial demonstration; passing all four supports an engineering decision. When a result fails,
preserve the counterexample and update the test suite before changing the implementation.

Finally, separate correctness, capability, efficiency, and safety conclusions. A correct
implementation may have weak capability; a capable prototype may be too costly or unsafe to
deploy. Name the population to which each conclusion applies and avoid converting a single
metric into a universal ranking. Track assumptions beside results, especially tokenizer and
template compatibility, data rights, access-control boundaries, and hardware-specific behavior.
Leave exercises with an executable acceptance criterion, a baseline result, and a short written
interpretation. That combination turns exploratory code into cumulative course evidence that can
be revisited when libraries, model families, or deployment engines change.


## 23.5 Sequence and logit distillation tradeoffs

Sequence distillation works across tokenizer boundaries and is storage-efficient, but discards the teacher's probability distribution. Logit distillation transfers dark knowledge but storing full vocabulary logits is expensive; top-k logits, online teachers, or selected positions trade fidelity for cost. Compare teacher-generated targets with ground truth and mix them deliberately. Temperature, KL direction, prompt masking, and tokenizer alignment are part of the objective and must be recorded.


In [ ]:
tokens,vocab,bytes_per=1_000_000,50_000,2
print("full-logit GiB",tokens*vocab*bytes_per/2**30)
for k in (8,32,128): print("top-k approximate GiB",tokens*k*(bytes_per+4)/2**30)


## 23.6 Teacher-error transfer audit

Build slices where the teacher is correct, incorrect, uncertain, stylistically biased, unsafe, or unable to follow the format. Measure whether the student reproduces each class before and after distillation. Independent verifiers and ground truth outrank teacher confidence. Multi-teacher agreement can reduce individual quirks but may suppress minority knowledge. Release lineage should identify teachers, revisions, prompts, sampling, filters, and licensing; distilled behavior does not erase upstream obligations.


In [ ]:
audit=[{"teacher":1,"student":1,"truth":1},{"teacher":0,"student":0,"truth":1},{"teacher":0,"student":1,"truth":1},{"teacher":1,"student":1,"truth":0}]
wrong=[r for r in audit if r["teacher"]!=r["truth"]]; print("teacher errors copied",sum(r["student"]==r["teacher"] for r in wrong),"of",len(wrong))


## 23.7 Current Hugging Face distillation workflows

Current TRL separates stable on-policy `DistillationTrainer` from experimental generalized knowledge-distillation workflows such as `GKDTrainer`. The stable trainer can use vLLM-powered student generation and a memory-efficient chunked Jensen–Shannon objective against teacher next-token distributions. APIs on the documentation's `main` branch may differ from the stable package, so pin TRL and record the exact class/config signature. Start with a tiny dry run, inspect generated student trajectories and teacher targets, and compare with ordinary sequence SFT under the same data and token budget. Online distillation increases systems complexity: teacher inference, generation workers, synchronization, cache behavior, and failure recovery belong in the experiment manifest and cost model.


In [ ]:
distillation_plan={"trl_version":"pin-stable","trainer":"DistillationTrainer","teacher":"teacher@commit","student":"student@commit","divergence":"chunked_jsd","generation_backend":"vllm","baseline":"sequence_sft"}
print(distillation_plan)
# Inspect the installed signature before constructing a real trainer:
try:
 import inspect,trl
 print("installed TRL",getattr(trl,"__version__","unknown"),"has stable trainer",hasattr(trl,"DistillationTrainer"))
 if hasattr(trl,"DistillationTrainer"): print(inspect.signature(trl.DistillationTrainer))
except ImportError: print("Install the notebook training extras for the live API inspection")


## Primary references and further study

Use the pinned library documentation that matches your environment. Papers explain the method and assumptions; current official documentation defines the executable API.

- [Distilling the Knowledge in a Neural Network](https://arxiv.org/abs/1503.02531)
- [Generalized Knowledge Distillation](https://arxiv.org/abs/2306.13649)
- [TRL trainers](https://huggingface.co/docs/trl/main/trainer)


## Exercises

    1. Implement temperature sweeps.
2. Verify and filter reasoning traces.
3. Calculate distillation break-even for a workload.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
